# RAG Step 6: Evaluation
Measure retrieval first, then answers.

In [ ]:
!pip install -q sentence-transformers chromadb

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
col = chromadb.PersistentClient(path="./chroma_store").get_collection("rag_demo")

def retrieve_sources(question, k=3):
    q = model.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    return [m["source"] for m in res["metadatas"][0]]

## 1. Build a small test set
Each question lists the source file that should be retrieved.

In [ ]:
testset = [
    ("What reduces hallucinations?",            "rag_overview.txt"),
    ("Which databases store embeddings?",       "vector_db.txt"),
    ("Why do chunks overlap?",                  "chunking.txt"),
    ("What is supervised learning?",            "ai_basics.txt"),
]

## 2. Retrieval metrics: Hit rate@k and MRR

In [ ]:
def evaluate(k=3):
    hits, rr = 0, 0.0
    for q, gold in testset:
        srcs = retrieve_sources(q, k)
        if gold in srcs:
            hits += 1
            rr += 1 / (srcs.index(gold) + 1)
    n = len(testset)
    return {"hit_rate": hits / n, "mrr": rr / n}

for k in (1, 3, 5):
    print(k, evaluate(k))

## 3. Answer-quality checks
Simple, cheap signals for generated answers.

In [ ]:
def faithfulness_proxy(answer, context):
    a, c = set(answer.lower().split()), set(context.lower().split())
    return len(a & c) / max(len(a), 1)   # share of answer words found in context

print(faithfulness_proxy("RAG reduces hallucinations", "RAG reduces hallucinations by grounding answers"))

## 4. What to track
- **Retrieval:** hit rate, MRR, recall@k
- **Generation:** faithfulness (is it supported by context?), answer relevance, "I don't know" correctness
- **System:** latency and cost per query

For larger projects, look at frameworks such as RAGAS or TruLens.

## Takeaways
Evaluate retrieval and generation separately so you know which part to fix.